# Get to Know a Dataset: GHS WUP DEGURBA

This notebook serves as a guided tour of the **GHS WUP DEGURBA** dataset.

More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

At the top level of our S3 bucket, we have a single key prefix **jrc-ghsl** that contains all GHSL datasets.

The prefix structure is the same for all datasets:

```
s3://jrc-ghsl/
  {product}/          # ghs-built-s, ghs-pop, ghs-smod, ...
    {release}/        # r2022a, r2023a, r2025a, ...
      {crs}/          # 54009, 4326
        {resolution}/ # 10m | 100m | 1000m | 3ss | 30ss
          {epoch}/    # 1975, 2018, 2030, ...
            {file}.tif
```

For example:

`ghs-wup-degurba/r2025a/54009/1000m/2020/GHS_WUP_DEGURBA_E2020_GLOBE_R2025A_54009_1000_V1_0.tif`

To quickly browse the GHS WUP DEGURBA datasets: https://jrc-ghsl.s3.amazonaws.com/index.html#ghs-wup-degurba/

Full documentation for this dataset can be found at: https://human-settlement.emergency.copernicus.eu/ghs_wup_degurba_r2025a.php

First we will import the Python libraries required throughout this notebook.

In [ ]:
# This notebook requires the following additional libraries
# (please install using the preferred method for your environment, e.g. pip, conda):
#
# boto3
# folium
# matplotlib
# pandas
# rasterio

# Import the libraries required for this notebook
# Installed libraries
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import folium
import matplotlib.pyplot as plt
import pandas as pd
import rasterio
from rasterio.session import AWSSession
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds


Next, we will define the location of our dataset, create our boto3 S3 client, and list the top level prefixes in our S3 bucket. Here we see there is only one top-level prefix in our bucket.

In [ ]:
# Location of the S3 bucket for this dataset
bucket = "jrc-ghsl"
prefix = "ghs-wup-degurba"

# List the top level of the bucket using boto3. Because this is a public bucket, we don't need to sign requests.
# Here we set the signature version to unsigned, which is required for public buckets.
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

# Print the items in the top-level prefixes
for item in s3.list_objects_v2(Bucket=bucket, Prefix=prefix, Delimiter='/')['CommonPrefixes']:
    print(item['Prefix'])

Looking into the top-level S3 prefix of our dataset, we list all available datasets.

In [ ]:
# display size in human readable format
def human_size(num_bytes):
    size = float(num_bytes)
    for unit in ("B", "KB", "MB", "GB", "TB", "PB"):
        if size < 1024 or unit == "PB":
            return f"{size:.1f} {unit}"
        size /= 1024

# List all keys as a table
s3 = boto3.resource("s3", config=Config(signature_version=UNSIGNED))
bucket = s3.Bucket("jrc-ghsl")
prefix = "ghs-wup-degurba"

df = pd.DataFrame(
    [(obj.key, obj.size, human_size(obj.size)) for obj in bucket.objects.filter(Prefix=prefix)],
    columns=["Key", "Bytes", "Size"],
)

# displays as a formatted table in Jupyter
pd.set_option("display.max_colwidth", None)  # None = no row content truncation;
(df.style
   .set_properties(subset=["Key"], **{"text-align": "left"})
   .set_table_styles([
       {"selector": "th.col_heading.level0.col0", "props": [("text-align", "left")]},
   ])
   .hide(axis="index")
)

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

Our dataset comes as a set of COG (Cloud Optimizied GeoTIFF) files organized as shown in the last section. A [COG](https://cogeo.org/) is a GeoTIFF whose metadata, internal tiling, and internal overviews (pyramids) are arranged to support efficient partial access over HTTP (Hypertext Transfer Protocol). Clients can issue HTTP range requests and read only the portions of the file they need.

Our dataset used this format because of these COG features:
 - **Partial reads over HTTP**: Clients fetch only the bytes for their area of interest via range requests, so users pull megabytes from S3 instead of downloading multi-GB global rasters
 - **Internal tiling**: Pixels are stored in square blocks, so a spatial query touches a small contiguous set of tiles rather than scanning the whole image
 - **Built-in overviews**: Embedded downsampled pyramids let zoomed-out views and previews load fast by reading low-resolution data
 - **Fully backward-compatible**: A COG is a valid GeoTIFF that any tool (GDAL, rasterio, QGIS, ArcGIS) reads with no special handling
 - **Ecosystem standard**: COG is the default for AWS Open Data, STAC, and the serverless geospatial stack, so data drops straight into existing pipelines and dynamic tiling services

COG is well supported by modern GIS software like [QGIS](https://www.qgis.org).

Packages such as [GDAL](https://gdal.org) and [rasterio](https://rasterio.readthedocs.io) can be used to work with COG in Python as well.

### Q: Can you show us an example of downloading and loading data from your dataset?

To demonstrate how to access the dataset, here is an example of how to download a COG from our S3 bucket and load its spatial structure using Python, boto3, and rasterio.

In [ ]:
# Unsigned access for the public bucket (no credentials needed)
aws_session = AWSSession(boto3.Session(), requester_pays=False)

# object key in the bucket
url = "s3://jrc-ghsl/ghs-wup-degurba/r2025a/54009/1000m/2020/GHS_WUP_DEGURBA_E2020_GLOBE_R2025A_54009_1000_V1_0.tif"

# unsigned_env makes GDAL skip request signing for the public bucket
with rasterio.Env(AWSSession(aws_unsigned=True)):
    with rasterio.open(url) as src:
        # Metadata only: no pixels transferred yet
        print("CRS (Projection):", src.crs)
        print("Size:            ", src.width, "x", src.height)
        print("Number of bands: ", src.count, "dtype:", src.dtypes[0])
        print("Resolution:      ", src.res)
        print("Bounds:          ", src.bounds)
        print("Overviews:       ", src.overviews(1))  # confirms it's a COG with pyramids

### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

To make visualizing our geospatial data smooth and high-performing, we separate the process into three logical steps:
 1. clip the data on a given bounding box
 2. static matrix distribution profiling
 3. interactive real-world mapping.

In [ ]:
bounds = 1006009, 4975949, 1075794, 5013628

with rasterio.Env(AWSSession(aws_unsigned=True)):
    with rasterio.open(url) as src:
        win = from_bounds(*bounds, transform=src.transform)
        data = src.read(1, window=win)
        src_crs = src.crs
        src_transform = src.transform
        src_bounds = src.bounds

print("Window shape:", data.shape)
print("min/max:", data.min(), data.max())
print("Coordinate Reference System:", src_crs)

Now let's create the plots that display the spatial representation and value distribution histogram

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm

# GHS WUP DEGURBA degree-of-urbanisation color table (same palette as GHS SMOD).
# Class value -> R G B A, 0-255. Classes not used are fully transparent (alpha 0).
degurba_rgba_255 = {
    10: (122, 182, 245, 255),  # water grid cell
    11: (205, 245, 122, 255),  # very low density rural grid cell
    12: (171, 205, 102, 255),  # low density rural grid cell
    13: (55, 86, 35, 255),     # rural cluster grid cell
    21: (255, 255, 0, 255),    # suburban / peri-urban grid cell
    22: (168, 112, 0, 255),    # semi-dense urban cluster grid cell
    23: (115, 38, 0, 255),     # dense urban cluster grid cell
    30: (255, 0, 0, 255),      # urban centre grid cell
}

# Build a ListedColormap indexed by class value 0..30 (transparent where unused)
max_class = 30
colors = []
for value in range(max_class + 1):
    r, g, b, a = degurba_rgba_255.get(value, (0, 0, 0, 0))
    colors.append((r / 255, g / 255, b / 255, a / 255))
degurba_cmap = ListedColormap(colors)
# Boundaries so each integer class maps to its own color slot
degurba_norm = BoundaryNorm(range(max_class + 2), degurba_cmap.N)

# Generate a side-by-side plot showcasing spatial attributes and data frequencies
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Spatial representation matrix using the DEGURBA color table
im = ax1.imshow(data, cmap=degurba_cmap, norm=degurba_norm)
ax1.set_title("GHS WUP DEGURBA Dataset Matrix Preview (Rome area)", fontweight='bold')
fig.colorbar(im, ax=ax1, label="Degree of Urbanisation Class Code")

# Plot 2: Value distribution histogram
ax2.hist(data.flatten(), bins=50, color='#3498db', edgecolor='white', alpha=0.8)
ax2.set_title("Data Value Distribution (Histogram)", fontweight='bold')
ax2.set_xlabel("Degree of Urbanisation Class Code")
ax2.set_ylabel("Occurrences Count")
# Show only the discrete DEGURBA class codes on the x-axis (not 12.5, 17.5, ...)
ax2.set_xticks(sorted(degurba_rgba_255))

plt.tight_layout()
plt.show()

Finally, let's create the interactive map that shot the entire dataset boundaries

In [ ]:
# Bounds in original dataset slightly exceed world limits
print(f'Bounds in {src_crs}: {src_bounds}')
left, bottom, right, top = src_bounds
left = -18000000 if left < -18000000 else left
right = 18000000 if right > 18000000 else right

# Convert bounds to EPSG:4326
dst_crs = "EPSG:4326"
left, bottom, right, top = transform_bounds(src_crs, dst_crs, left, bottom, right, top, densify_pts=21)
print(f'Bounds in {dst_crs}: {[left, bottom, right, top]}')

# Initialize an interactive leafet map focused right over dataset's center coordinates
center_lat = (bottom + top) / 2
center_lon = (left + right) / 2
m = folium.Map(location=[center_lat, center_lon], zoom_start=1, tiles="OpenStreetMap")

# Overlay a bounding perimeter highlighting exactly where this raster dataset sits globally
folium.Rectangle(
    bounds=[[bottom, left], [top, right]],
    color="#e74c3c",
    weight=3,
    fill=True,
    fill_color="#e74c3c",
    fill_opacity=0.15,
    popup="Dataset Coverage Limit"
).add_to(m)

# Display interactive visualization map
m

<!-- DATA PROVIDER INSTRUCTIONS
This section is less prescriptive / freeform than previous sections. The goal here is to show an opinionated example of answering a question using your data. The scale of your dataset may preclude a full example, and so feel free to limit the scope of this example (e.g. work on a subset of data). Users should be able to replicate your example in this notebook, and get a sense of how they would scale up.

A "toy" example is better than no example.

Ideally, your example would:
1. Transmit some of your domain & dataset experience to the reader, drawing on your own work as much as possible
2. Provide a jumping off point for users to extend your work, and do novel work of their own.

DATA PROVIDER INSTRUCTIONS -->

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

EXAMPLE - REPLACE

Lorem ipsum dolor sit amet, consectetur adipiscing elit. Praesent sem enim, finibus vel leo vel, blandit interdum tortor. In lectus dolor, congue eu odio vel, euismod facilisis lacus. Vestibulum aliquet, quam in rhoncus tempus, arcu massa suscipit lacus, nec fermentum quam justo nec lacus. Duis id leo fermentum ante tempor pulvinar eu vitae ligula. Cras feugiat vel ligula sit amet lacinia. Mauris sit amet sem vestibulum ligula volutpat iaculis in eu velit. Sed turpis magna, porta ac nisi vitae, maximus volutpat mi. Vestibulum mattis est eros, nec pellentesque nisi iaculis sed. 

EXAMPLE - REPLACE

<!-- DATA PROVIDER INSTRUCTIONS
This section is, like the previous one, intended to be freeform / non-prescriptive. The goal here is to provide a challenge to the community to do something novel with your dataset. That can either be novel in terms of the task, or novel in terms of methodological or computational approach.

Another way to consider this section, is as a wishlist. If you were less constrained by time, cost, skill, etc., what would you like to see achieved using these data? 

The challenge should, however, be somewhat realistic. A challenge that assumes e.g. original data collection, is likely to go unanswered.
DATA PROVIDER INSTRUCTIONS -->

### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

EXAMPLE - REPLACE

Lorem ipsum dolor sit amet, consectetur adipiscing elit. Praesent sem enim, finibus vel leo vel, blandit interdum tortor. In lectus dolor, congue eu odio vel, euismod facilisis lacus. Vestibulum aliquet, quam in rhoncus tempus, arcu massa suscipit lacus, nec fermentum quam justo nec lacus. Duis id leo fermentum ante tempor pulvinar eu vitae ligula. Cras feugiat vel ligula sit amet lacinia. Mauris sit amet sem vestibulum ligula volutpat iaculis in eu velit. Sed turpis magna, porta ac nisi vitae, maximus volutpat mi. Vestibulum mattis est eros, nec pellentesque nisi iaculis sed. 

EXAMPLE - REPLACE

# DATA PROVIDER: PLEASE REMEMBER TO CLEAR ALL OUTPUTS BEFORE COMMITTING TO YOUR GITHUB REPOSITORY